In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# AI Transpilerパス
AI搭載Transpilerパスは、一部のトランスパイル処理において「従来の」Qiskitパスのドロップイン置き換えとして機能するパスです。既存のヒューリスティックアルゴリズムよりも優れた結果（低い深度やCNOTカウントなど）を生成することが多く、ブール充足可能性ソルバーなどの最適化アルゴリズムよりもはるかに高速です。AI Transpilerパスは、ローカル環境で実行するか、IBM Quantum&reg; Premium Plan、Flex Plan、またはOn-Prem（IBM Quantum Platform API経由）Planに加入している場合はQiskit Transpiler Serviceを使用してクラウド上で実行できます。

> **Note:** AI搭載Transpilerパスはベータリリース状態であり、変更される可能性があります。
>     フィードバックや開発チームへの連絡は、こちらの[Qiskit Slack Workspaceチャンネル](https://qiskit.slack.com/archives/C06KF8YHUAU)をご利用ください。

現在、以下のパスが利用可能です：

**ルーティングパス**
 - `AIRouting`: レイアウト選択と回路ルーティング

**回路合成パス**
 - `AICliffordSynthesis`: Clifford回路合成
 - `AILinearFunctionSynthesis`: 線形関数回路合成
 - `AIPermutationSynthesis`: 置換回路合成
 - `AIPauliNetworkSynthesis`: Pauli Network回路合成

AI Transpilerパスを使用するには、まず[`qiskit-ibm-transpiler`パッケージをインストール](/guides/qiskit-transpiler-service#install-transpiler-service)してください。利用可能なさまざまなオプションの詳細については、[qiskit-ibm-transpiler APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler)をご覧ください。

## AI Transpilerパスをローカルまたはクラウドで実行する
AI搭載Transpilerパスをローカル環境で無料で使用したい場合は、追加の依存関係を含めて`qiskit-ibm-transpiler`を以下のようにインストールしてください：

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

これらの追加依存関係がない場合、AI搭載TranspilerパスはQiskit Transpiler Serviceを通じてクラウド上で実行されます（IBM Quantum Premium Plan、Flex Plan、またはOn-Prem（IBM Quantum Platform API経由）Planユーザーのみ利用可能）。追加依存関係をインストールした後は、AI搭載Transpilerパスのデフォルトの実行モードはローカルマシンの使用になります。

## AIルーティングパス
`AIRouting`パスは、レイアウトステージとルーティングステージの両方として機能します。以下のように`PassManager`内で使用できます：

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

ここで、`backend`はルーティング対象のカップリングマップを決定し、`optimization_level`（1、2、または3）は処理に費やす計算量を決定し（高いほど通常は良い結果が得られますが、時間がかかります）、`layout_mode`はレイアウト選択の処理方法を指定します。
`layout_mode`には以下のオプションがあります：

- `keep`: 前のTranspilerパスで設定されたレイアウトを尊重します（設定されていない場合はトリビアルレイアウトを使用します）。通常、デバイスの特定のQubitで回路を実行する必要がある場合にのみ使用されます。最適化の余地が少ないため、結果が悪くなることが多いです。
- `improve`: 前のTranspilerパスで設定されたレイアウトを開始点として使用します。レイアウトの良い初期推定がある場合に便利です。例えば、デバイスのカップリングマップにおおよそ従うように構築された回路の場合です。また、他の特定のレイアウトパスを`AIRouting`パスと組み合わせて試したい場合にも便利です。
- `optimize`: デフォルトモードです。レイアウトの良い推定がない一般的な回路に最適です。このモードは前のレイアウト選択を無視します。
- `local_mode`: このフラグは`AIRouting`パスの実行場所を決定します。`False`の場合、`AIRouting`はQiskit Transpiler Serviceを通じてリモートで実行されます。`True`の場合、パッケージはローカル環境でパスを実行しようとし、必要な依存関係が見つからない場合はクラウドモードにフォールバックします。

## AI回路合成パス
AI回路合成パスを使用すると、さまざまな回路タイプ（[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford)、[Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction)、[Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation)、Pauli Network）の部分を再合成して最適化できます。合成パスの典型的な使用方法は以下の通りです：

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

合成はデバイスのカップリングマップを尊重します。他のルーティングパスの後に安全に実行でき、回路を乱すことなく、全体の回路がデバイスの制約に従い続けます。デフォルトでは、合成されたサブ回路が元のサブ回路を改善する場合にのみ置き換えが行われます（現在はCNOTカウントのみをチェック）。ただし、`replace_only_if_better=False`を設定することで、常に回路を置き換えるように強制できます。

`qiskit_ibm_transpiler.ai.synthesis`から以下の合成パスが利用可能です：

- *AICliffordSynthesis*: [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford)回路（`H`、`S`、`CX`ゲートのブロック）の合成。現在、最大9 Qubitブロックに対応しています。
- *AILinearFunctionSynthesis*: [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction)回路（`CX`および`SWAP`ゲートのブロック）の合成。現在、最大9 Qubitブロックに対応しています。
- *AIPermutationSynthesis*: [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation)回路（`SWAP`ゲートのブロック）の合成。現在、65、33、27 Qubitブロックに対応しています。
- *AIPauliNetworkSynthesis*: Pauli Network回路（`H`、`S`、`SX`、`CX`、`RX`、`RY`、`RZ`ゲートのブロック）の合成。現在、最大6 Qubitブロックに対応しています。

サポートされるブロックサイズは段階的に拡大していく予定です。

すべてのパスはスレッドプールを使用して複数のリクエストを並列に送信します。デフォルトでは、最大スレッド数はコア数に4を加えた値です（Python `ThreadPoolExecutor`オブジェクトのデフォルト値）。ただし、パスのインスタンス化時に`max_threads`引数で独自の値を設定できます。例えば、以下の行は最大20スレッドを使用できる`AILinearFunctionSynthesis`パスをインスタンス化します。